In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

In [0]:
from pyspark.sql import functions as F
from datetime import date, timedelta
import requests

hoje = date.today()
dbutils.widgets.text("data_inicio", (hoje - timedelta(days=7)).strftime("%m-%d-%Y"))
dbutils.widgets.text("data_fim", hoje.strftime("%m-%d-%Y"))

BASE = "/Volumes/workspace/bronze/inputs/"

In [0]:
arquivos = {
    "movies_info_TMDB_IMDB.csv":       "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv":    "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv":  "bronze.tb_credits_and_tags",
    "movies_reviews.csv":              "bronze.tb_movies_reviews",
}

for arquivo, tabela in arquivos.items():
    df = (spark.read
          .option("header", True)
          .option("quote", '"')
          .option("escape", '"')
          .csv(BASE + arquivo))
    df = df.withColumn("ingestion_datetime", F.current_timestamp())
    df.write.format("delta").mode("append").saveAsTable(tabela)
    print(tabela, "->", df.count(), "linhas")

bronze.tb_movies_info -> 106930 linhas
bronze.tb_movies_financials -> 106165 linhas
bronze.tb_movies_metrics -> 107364 linhas
bronze.tb_credits_and_tags -> 106320 linhas
bronze.tb_movies_reviews -> 32412 linhas


In [0]:
import json

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

url = ("https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
       "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
       f"?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
       "&$select=dataHoraCotacao,cotacaoCompra&$format=json")

# tenta a API; no Free Edition o domínio do BCB é bloqueado, então cai no arquivo de backup
try:
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    dados = resp.json()["value"]
    origem = "API do BCB"
except requests.exceptions.RequestException as e:
    print("API indisponível neste ambiente:", type(e).__name__)
    with open(BASE + "cotacao_dolar.json") as f:
        dados = json.load(f)["value"]
    origem = "arquivo cotacao_dolar.json (backup)"

print("origem:", origem, "| cotações:", len(dados))

if dados:
    df_cot = (spark.createDataFrame(dados, "dataHoraCotacao string, cotacaoCompra double")
              .withColumn("ingestion_datetime", F.current_timestamp()))
    df_cot.write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar")
    display(df_cot)

API indisponível neste ambiente: ConnectionError
origem: arquivo cotacao_dolar.json (backup) | cotações: 20


dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-08-21 13:07:28.20125,5.1619,2026-09-21T12:23:49.658Z
2026-08-24 13:04:14.216099,5.1506,2026-09-21T12:23:49.658Z
2026-08-25 13:04:44.743388,5.1484,2026-09-21T12:23:49.658Z
2026-08-26 13:05:20.847747,5.1598,2026-09-21T12:23:49.658Z
2026-08-27 13:10:22.548239,5.1637,2026-09-21T12:23:49.658Z
2026-08-28 13:11:19.963529,5.1999,2026-09-21T12:23:49.658Z
2026-08-31 13:08:57.200562,5.181,2026-09-21T12:23:49.658Z
2026-09-01 13:06:03.857125,5.1564,2026-09-21T12:23:49.658Z
2026-09-02 13:02:37.601302,5.1267,2026-09-21T12:23:49.658Z
2026-09-03 13:06:38.232974,5.0956,2026-09-21T12:23:49.658Z
